# 🔧 Notebook 01 — Environment Setup & Verification
**LSTM + DQN Backtesting Suite**

This notebook:
- Installs all required packages
- Verifies GPU / CUDA availability
- Checks minimum version requirements
- Sets global reproducibility seeds


## 1. Install Requirements

In [11]:
!nvidia-smi

Wed Jun  3 02:19:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 Ti     Off |   00000000:01:00.0 Off |                  N/A |
| N/A   65C    P0             17W /   50W |     505MiB /   4096MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
# Install all dependencies from requirements_dl.txt
import subprocess, sys

def pip_install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])

core_packages = [
    'numpy>=1.24.0', 'pandas>=2.0.0', 'scipy>=1.11.0',
    'torch>=2.1.0', 'torchvision>=0.16.0',
    'stable-baselines3>=2.2.0', 'gymnasium>=0.29.0', 'shimmy>=1.3.0',
    'yfinance>=0.2.36', 'pandas-datareader>=0.10.0',
    'alpha-vantage>=2.3.1', 'ccxt>=4.2.0', 'nsepy>=0.8',
    'ta>=0.11.0', # 'pandas-ta>=0.3.14b',  # DISABLED: requires numba (Python <3.14 only) 'stockstats>=0.6.2',
    'backtesting>=0.3.3', # 'vectorbt>=0.26.1',     # DISABLED: requires numba (Python <3.14 only)
    'pyfolio-reloaded>=0.9.5', 'quantstats>=0.0.62',
    'scikit-learn>=1.3.0', 'matplotlib>=3.8.0',
    'seaborn>=0.13.0', 'plotly>=5.18.0',
    'ipywidgets>=8.1.0', 'tqdm>=4.66.0',
    'pyyaml>=6.0.1', 'python-dotenv>=1.0.0',
    'loguru>=0.7.2', 'joblib>=1.3.2', 'gputil>=1.4.0', 'psutil>=5.9.6'
]

print('📦 Installing packages...')
for pkg in core_packages:
    try:
        pip_install(pkg)
        print(f'  ✅ {pkg.split(">")[0]}')
    except Exception as e:
        print(f'  ⚠️  {pkg.split(">")[0]} — {e}')
print('\n✨ Installation complete!')

📦 Installing packages...
  ✅ numpy
  ✅ pandas
  ✅ scipy
  ✅ torch
  ✅ torchvision
  ✅ stable-baselines3
  ✅ gymnasium
  ✅ shimmy
  ✅ yfinance
  ✅ pandas-datareader
  ✅ alpha-vantage
  ✅ ccxt
  ✅ nsepy
  ✅ ta
  ✅ backtesting
  ✅ pyfolio-reloaded
  ✅ quantstats
  ✅ scikit-learn
  ✅ matplotlib
  ✅ seaborn
  ✅ plotly
  ✅ ipywidgets
  ✅ tqdm
  ✅ pyyaml
  ✅ python-dotenv
  ✅ loguru
  ✅ joblib
  ✅ gputil
  ✅ psutil

✨ Installation complete!


## 2. Import & Version Check

In [13]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
import platform
import datetime
from pathlib import Path

# ---- Core ----
import numpy as np
import pandas as pd
import scipy

# ---- Deep Learning ----
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ---- RL ----
import gymnasium as gym
import stable_baselines3 as sb3
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env

# ---- Data Sources ----
import yfinance as yf
import ccxt

# ---- Technical Analysis ----
import ta
# import pandas_ta as pta  # DISABLED: requires numba (Python <3.14)

# ---- Backtesting ----
# import vectorbt as vbt   # DISABLED: requires numba (Python <3.14)
import quantstats as qs

# ---- ML ----
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

# ---- Visualisation ----
import matplotlib
import plotly
import seaborn

# ---- Utilities ----
import yaml
from tqdm import tqdm
from loguru import logger

print('=' * 60)
print('  🧠 LSTM + DQN Backtesting Suite — Environment Report')
print('=' * 60)
print(f'  Python       : {sys.version.split()[0]}')
print(f'  Platform     : {platform.system()} {platform.release()}')
print(f'  Timestamp    : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()
print('  📦 Package Versions:')
packages = {
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'stable-baselines3': sb3.__version__,
    'gymnasium': gym.__version__,
    'yfinance': yf.__version__,
    'sklearn': sklearn.__version__,
    'plotly': plotly.__version__,
    # 'vectorbt': vbt.__version__,  # DISABLED
}
for pkg, ver in packages.items():
    print(f'    {pkg:<20} {ver}')
print()

  🧠 LSTM + DQN Backtesting Suite — Environment Report
  Python       : 3.14.5
  Platform     : Linux 7.0.10-arch1-1
  Timestamp    : 2026-06-03 02:19:56

  📦 Package Versions:
    numpy                2.4.2
    pandas               2.3.3
    torch                2.12.0+cu130
    stable-baselines3    2.8.0
    gymnasium            1.2.3
    yfinance             1.2.0
    sklearn              1.8.0
    plotly               6.7.0



## 3. GPU / CUDA Detection

In [14]:
import psutil

print('  🖥️  Hardware:')
print(f'    CPU cores    : {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical')
ram = psutil.virtual_memory()
print(f'    RAM          : {ram.total / (1024**3):.1f} GB total, {ram.available / (1024**3):.1f} GB available')
print()

# ---- PyTorch GPU ----
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()

if cuda_available:
    device = torch.device('cuda')
    print(f'  🚀 CUDA GPU detected!')
    print(f'    Device       : {torch.cuda.get_device_name(0)}')
    print(f'    CUDA version : {torch.version.cuda}')
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f'    VRAM         : {vram / (1024**3):.1f} GB')
elif mps_available:
    device = torch.device('mps')
    print('  🍎 Apple MPS GPU detected! (Metal Performance Shaders)')
else:
    device = torch.device('cpu')
    print('  ⚠️  No GPU detected — running on CPU (training will be slower)')

print(f'\n  ✅ Using device: {device}')
print('=' * 60)

  🖥️  Hardware:
    CPU cores    : 4 physical, 8 logical
    RAM          : 7.5 GB total, 1.2 GB available

  🚀 CUDA GPU detected!
    Device       : NVIDIA GeForce GTX 1650 Ti
    CUDA version : 13.0
    VRAM         : 3.6 GB

  ✅ Using device: cuda


## 4. Set Global Seeds for Reproducibility

In [15]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if cuda_available:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'✅ All random seeds set to {SEED} for reproducibility.')

✅ All random seeds set to 42 for reproducibility.


## 5. Load Configuration Files

In [16]:
BASE_DIR = Path('.').resolve().parent if Path('.').resolve().name == 'deep_learning_backtest' else Path('.').resolve()
CONFIG_DIR = BASE_DIR / 'deep_learning_backtest' / 'configs'

with open(CONFIG_DIR / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)
    
with open(CONFIG_DIR / 'data_config.yaml') as f:
    data_cfg = yaml.safe_load(f)

print('✅ model_config.yaml loaded')
print('✅ data_config.yaml loaded')
print(f"\n  Symbols: {data_cfg['symbols']['equities']}")
print(f"  Train: {data_cfg['train_start']} → {data_cfg['train_end']}")
print(f"  Test:  {data_cfg['test_start']}  → {data_cfg['test_end']}")
print(f"  LSTM hidden_size: {model_cfg['lstm']['hidden_size']}")
print(f"  DQN memory_size:  {model_cfg['dqn']['memory_size']}")

✅ model_config.yaml loaded
✅ data_config.yaml loaded

  Symbols: ['AAPL', 'MSFT', 'GOOGL', 'SPY', 'QQQ', 'TSLA']
  Train: 2012-01-01 → 2023-12-31
  Test:  2024-01-01  → 2026-05-31
  LSTM hidden_size: 64
  DQN memory_size:  100000
